In [0]:
WITH rfm_csv AS (
  SELECT *
  FROM read_files(
    'file:/Workspace/Users/mstaiman@hbse.com/rfm-segmentation-test/datasets/01_RFM_Segmentation_email_output_08062026.csv',
    format => 'csv',
    header => true
  )
)
SELECT
  c.*,
  d.* EXCEPT (d.audienceid)
FROM rfm_csv c
LEFT JOIN kagr_njd.stage.dim_aggregatefields d
  ON c.audienceid = d.audienceid;



-- General null count per column across the joined dataset, without hardcoding any column names.
-- Uses to_json/from_json against struct(*) (same technique as the per-row null count below),
-- then LATERAL VIEW explode() to pivot every column into (column_name, value) pairs to aggregate over.
WITH rfm_csv AS (
  SELECT *
  FROM read_files(
    'file:/Workspace/Users/mstaiman@hbse.com/rfm-segmentation-test/datasets/01_RFM_Segmentation_email_output_08062026.csv',
    format => 'csv',
    header => true
  )
),
joined AS (
  SELECT c.*, d.* EXCEPT (d.audienceid)
  FROM rfm_csv c
  LEFT JOIN kagr_njd.stage.dim_aggregatefields d
    ON c.audienceid = d.audienceid
),
row_maps AS (
  SELECT
    from_json(
      to_json(struct(*), map('ignoreNullFields', 'false')),
      'map<string,string>'
    ) AS col_map
  FROM joined
),
exploded AS (
  SELECT column_name, value
  FROM row_maps
  LATERAL VIEW explode(col_map) exploded_map AS column_name, value
)
SELECT
  column_name,
  SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS null_count
FROM exploded
GROUP BY column_name
HAVING SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) > 0
ORDER BY null_count DESC;


-- Count how many columns are null for each individual row (customer) after the join.
-- Uses to_json/from_json against struct(*) so it works dynamically without hardcoding every column name.
WITH rfm_csv AS (
  SELECT *
  FROM read_files(
    'file:/Workspace/Users/mstaiman@hbse.com/rfm-segmentation-test/datasets/01_RFM_Segmentation_email_output_08062026.csv',
    format => 'csv',
    header => true
  )
),
joined AS (
  SELECT c.*, d.* EXCEPT (d.audienceid)
  FROM rfm_csv c
  LEFT JOIN kagr_njd.stage.dim_aggregatefields d
    ON c.audienceid = d.audienceid
),
row_nulls AS (
  SELECT
    audienceid,
    emailaddress,
    from_json(
      to_json(struct(*), map('ignoreNullFields', 'false')),
      'map<string,string>'
    ) AS col_map
  FROM joined
)
SELECT
  audienceid,
  emailaddress,
  size(col_map) AS total_columns,
  size(filter(map_values(col_map), x -> x IS NULL)) AS null_column_count
FROM row_nulls
ORDER BY null_column_count DESC;